# Kiểm chứng và kết luận MixStyle — 10/11/2026

Notebook này hoàn tất phân tích của 12 checkpoint và 57 capture ngoại bộ đã lưu.
Notebook 10 được giữ làm lịch sử: lần tổng hợp đầu lỗi do baseline có thêm dòng
ngưỡng mặc định 0,5. Phân tích hiện tại lọc đúng `source_calibrated`.
Mã runner cũ được khôi phục và đối chiếu đúng SHA256 trong `provenance/`.

Không chọn lại mô hình theo test. Probe và runtime dùng checkpoint đã đóng băng.
Đây là nghiên cứu hậu nghiệm, không thay kết quả chính của khóa luận.


In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p/'code/scripts/run_mixstyle_study.py').is_file())
OUT = ROOT/'code/results/mixstyle_posthoc_2026-11-09'
def run(*args):
    result = subprocess.run([sys.executable, '-X', 'utf8', *args], cwd=ROOT,
                            text=True, encoding='utf-8', stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()
run('-m', 'pytest', 'code/tests/test_mixstyle.py', '-q',
    '--junitxml='+str(OUT/'mixstyle_tests_final.xml'))


In [ ]:
# Recompute primary metrics and paired intervals from frozen predictions.
run('code/scripts/run_mixstyle_study.py', '--stage', 'summarize', '--output', str(OUT))


In [ ]:
# Reuse hash-verified completed secondary work; otherwise execute it.
run('code/scripts/complete_mixstyle_secondary.py')
run('code/scripts/report_mixstyle_results.py')


In [ ]:
import pandas as pd
display(pd.read_csv(OUT/'analysis/six_method_equal_set.csv').query("condition == 'B'"))
display(pd.read_csv(OUT/'analysis/probe_means.csv').query("control == 'learned'"))
display(pd.read_csv(OUT/'analysis/runtime_remeasured.csv'))
print(json.loads((OUT/'analysis/final_report.json').read_text(encoding='utf-8')))
print((OUT/'analysis/BAO_CAO_MIXSTYLE.txt').read_text(encoding='utf-8'))
